In [1]:
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, min, round, avg, count, sum, when
from pyspark.sql.types import IntegerType

StatementMeta(, 4508dc45-b386-4072-946e-56134b8853d9, 5, Finished, Available, Finished, False)

In [2]:
# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

today_str = datetime.now().strftime("%Y-%m-%d")

StatementMeta(, 4508dc45-b386-4072-946e-56134b8853d9, 6, Finished, Available, Finished, False)

# Price data tranformation

In [3]:



source_table = "stg_prices" # The table we built in the last step
target_table = "int_price_transformed"

print(f"Starting Price Transformation for {today_str}...")

try:
    # 1. Read the Silver table
    df_silver = spark.read.format("delta").table(source_table)

    # 2. Filter for today's extraction batch 
    # (This prevents Spark from recalculating 100-day averages for all of history every day)
    df_today_batch = df_silver.filter(col("extract_date") == today_str)

    # 3. Perform the GroupBy and Aggregations
    df_transformed = df_today_batch.groupBy("extract_date", "symbol").agg(
        round(avg("close_price"), 2).alias("avg_close_price_past_100days"),
        max("high_price").alias("max_price_past_100days"),
        min("low_price").alias("min_price_past_100days"),
        round(avg("volume"), 0).alias("avg_volume_past_100days"), # Rounding volume to whole numbers
        max("volume").alias("max_volume_past_100days"),
        min("volume").alias("min_volume_past_100days")
    )

    # 4. The Time & Resource Saver Logic (Pre-Filter and Fast Append)
    if spark.catalog.tableExists(target_table):
        
        # A. Query the target table to see which symbols we already transformed today
        loaded_df = spark.sql(f"SELECT DISTINCT symbol FROM {target_table} WHERE extract_date = '{today_str}'")
        loaded_symbols = [row['symbol'] for row in loaded_df.collect()]
        
        # B. Filter out already processed symbols
        if loaded_symbols:
            print(f"Symbols already transformed for {today_str}: {loaded_symbols}")
            df_transformed = df_transformed.filter(~col("symbol").isin(loaded_symbols))
            
        # C. Check if we have any data left to write
        if df_transformed.isEmpty():
            print(f"⏭️ Transformed price data already exists for {today_str}. Skipping write!")
        else:
            new_record_count = df_transformed.count()
            print(f"Executing fast APPEND for {new_record_count} transformed records...")
            
            (df_transformed.write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table))
            
            print(f"✅ Successfully appended transformed prices to {target_table}")

    else:
        # First-time run: Create the table
        print(f"Table does not exist. Creating {target_table}...")
        (df_transformed.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print(f"✅ Successfully created {target_table}")

    # Display the final transformed row(s) for today
    display(spark.sql(f"SELECT * FROM {target_table} WHERE extract_date = '{today_str}'"))

except Exception as e:
    print(f"❌ Transformation failed. Error: {e}")

StatementMeta(, 4508dc45-b386-4072-946e-56134b8853d9, 7, Finished, Available, Finished, False)

Starting Price Transformation for 2026-06-17...
Executing fast APPEND for 3 transformed records...
✅ Successfully appended transformed prices to int_price_transformed


SynapseWidget(Synapse.DataFrame, 344d201f-7de3-4099-93b7-90229a78e5f3)

# News data tranformation

In [4]:

source_table = "stg_news"
target_table = "int_news_transformed"

print(f"Starting News Transformation for {today_str}...")

try:
    # 1. Read the Silver table
    df_silver_news = spark.read.format("delta").table(source_table)

    # 2. Filter for today's extraction batch
    df_today_news = df_silver_news.filter(col("extract_date") == today_str)

    # 3. Perform the GroupBy and Conditional Aggregations
    df_transformed = df_today_news.groupBy("extract_date", "symbol").agg(
        count("*").alias("total_articles"),
        
        # Conditional sums to act as specific counts
        sum(when(col("overall_sentiment_label") == "Bullish", 1).otherwise(0)).alias("bullish_count"),
        sum(when(col("overall_sentiment_label") == "Somewhat-Bullish", 1).otherwise(0)).alias("somewhat_bullish_count"),
        sum(when(col("overall_sentiment_label") == "Neutral", 1).otherwise(0)).alias("neutral_count"),
        sum(when(col("overall_sentiment_label") == "Somewhat-Bearish", 1).otherwise(0)).alias("somewhat_bearish_count"),
        sum(when(col("overall_sentiment_label") == "Bearish", 1).otherwise(0)).alias("bearish_count"),
        
        # Calculate the average sentiment score and round it for clean reporting
        round(avg("overall_sentiment_score"), 4).alias("avg_sentiment_score")
    )

    # 4. The Time & Resource Saver Logic (Pre-Filter and Fast Append)
    if spark.catalog.tableExists(target_table):
        
        # A. Query to see which symbols we already transformed today
        loaded_df = spark.sql(f"SELECT DISTINCT symbol FROM {target_table} WHERE extract_date = '{today_str}'")
        loaded_symbols = [row['symbol'] for row in loaded_df.collect()]
        
        # B. Filter out already processed symbols
        if loaded_symbols:
            print(f"Symbols already transformed for {today_str}: {loaded_symbols}")
            df_transformed = df_transformed.filter(~col("symbol").isin(loaded_symbols))
            
        # C. Check if we have any data left to write
        if df_transformed.isEmpty():
            print(f"⏭️ Transformed news data already exists for {today_str}. Skipping write!")
        else:
            new_record_count = df_transformed.count()
            print(f"Executing fast APPEND for {new_record_count} transformed records...")
            
            (df_transformed.write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table))
            
            print(f"✅ Successfully appended transformed news to {target_table}")

    else:
        # First-time run: Create the table
        print(f"Table does not exist. Creating {target_table}...")
        (df_transformed.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print(f"✅ Successfully created {target_table}")

    # Display the final transformed row(s) for today
    display(spark.sql(f"SELECT * FROM {target_table} WHERE extract_date = '{today_str}'"))

except Exception as e:
    print(f"❌ Transformation failed. Error: {e}")

StatementMeta(, 4508dc45-b386-4072-946e-56134b8853d9, 8, Finished, Available, Finished, False)

Starting News Transformation for 2026-06-17...
Executing fast APPEND for 3 transformed records...
✅ Successfully appended transformed news to int_news_transformed


SynapseWidget(Synapse.DataFrame, cad71483-abd8-421b-bdc8-e0fc0aed633f)

# Biz Info. Transformation

In [ ]:
# Define your target table
table_name = "business_info"

print(f"Starting schema transformation for {table_name}...")

try:
    # 1. Read the existing Delta table into a dataframe
    df_existing = spark.read.format("delta").table(table_name)
    
    # 2. Cast the text columns to integers
    df_transformed = df_existing \
        .withColumn("AnalystRatingBuy", col("AnalystRatingBuy").cast(IntegerType())) \
        .withColumn("AnalystRatingHold", col("AnalystRatingHold").cast(IntegerType())) \
        .withColumn("AnalystRatingSell", col("AnalystRatingSell").cast(IntegerType())) \
        .withColumn("AnalystRatingStrongBuy", col("AnalystRatingStrongBuy").cast(IntegerType())) \
        .withColumn("AnalystRatingStrongSell", col("AnalystRatingStrongSell").cast(IntegerType()))
        
    print("✅ Columns successfully cast to IntegerType.")

    # 3. Write back to the exact same table with 'overwriteSchema' enabled
    (df_transformed.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true") 
        .saveAsTable(table_name))
        
    print(f"✅ Successfully updated schema and overwritten {table_name}!")
    
    # 4. Verify the new schema
    display(spark.sql(f"DESCRIBE {table_name}"))

except Exception as e:
    print(f"❌ Failed to transform schema. Error: {e}")